In [1]:
##### GPU is required to run this kernel (Ran on Kaggle GPU originally)

In [2]:
import numpy as np
import pandas as pd 
from sklearn.model_selection import *
from sklearn.preprocessing import *
from sklearn.metrics import *
from lightgbm import LGBMClassifier
import gc

from tqdm import tqdm, tqdm_notebook

import warnings
warnings.simplefilter('ignore')


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import datetime
import time

In [3]:
# import multiprocessing as mp
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model, Sequential, load_model
from tensorflow.keras.optimizers import *
from tensorflow.keras import regularizers
from tensorflow.keras.activations import *
import tensorflow as tf

In [5]:
train = pd.read_csv('Train.csv')
test = pd.read_csv('Test.csv')
ss = pd.read_csv('SampleSubmission.csv')

In [6]:
prod_cols = ['P5DA', 'RIBP', '8NN1','7POT', '66FJ', 'GYSR', 'SOP4', 'RVSZ', 'PYUQ', 'LJR9', 'N2MW', 'AHXO',
             'BSTQ', 'FM3X', 'K6QO', 'QBOL', 'JWFN', 'JZ9D', 'J9JW', 'GHYX', 'ECY3']

In [7]:
ID_COL, TARGET_COL = 'ID', 'target'

In [8]:
miss_prod_lst = []
for p in prod_cols:
    tmp_df = train[train[p] == 1]
    tmp_df['target'] = p
    tmp_df[p] = 0  
    miss_prod_lst.append(tmp_df)
train = pd.concat(miss_prod_lst, axis=0)

In [9]:
train = train.sample(frac=1, random_state=2)
mapper = pd.Series(index = train['target'].unique(), data = np.arange(train['target'].nunique())).to_dict()
rev_mapper = {val: key for key, val in mapper.items()}
train['target'] = train['target'].map(mapper)

In [10]:
def comb_cols(c1, c2):
    return df[c1].astype('str') + df[c2].astype('str')

In [11]:
train

,ID,join_date,sex,marital_status,birth_year,branch_code,occupation_code,occupation_category_code,P5DA,RIBP,...,BSTQ,FM3X,K6QO,QBOL,JWFN,JZ9D,J9JW,GHYX,ECY3,target
26640,8ZJS7MB,1/12/2019,M,M,1968,748L,WMTK,90QI,0,0,...,0,0,1,0,0,0,0,0,0,0
11384,WLRIN0K,1/6/2019,M,M,1983,E5SW,UJ5T,T4MS,0,0,...,0,0,0,1,0,0,0,0,0,1
3171,DJM35MX,1/4/2019,M,M,1971,UAOD,2A7I,T4MS,0,0,...,0,0,1,0,0,0,0,0,0,2
26443,FNFXB0F,1/7/2019,M,M,1979,XX25,N2ZZ,90QI,0,0,...,0,0,0,0,0,0,0,0,0,1
24528,GJ09RVA,1/7/2020,M,M,1962,94KC,2A7I,T4MS,0,0,...,0,0,0,0,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28509,JU2R9UG,1/6/2018,M,M,1961,E5SW,2A7I,T4MS,0,0,...,0,0,0,1,0,0,0,0,0,1
15584,PS9LTQ0,1/5/2018,F,M,1967,XX25,SST3,56SI,0,0,...,0,0,0,0,0,0,0,0,0,1
1505,3MGR8S6,1/7/2017,F,W,1973,30H5,2A7I,T4MS,0,0,...,0,0,0,0,0,0,0,0,0,1
3014,BHLIADF,1/3/2012,M,U,1978,748L,0OJM,T4MS,0,1,...,0,0,0,0,0,0,0,0,0,14


In [12]:
df = pd.concat([train, test]).reset_index(drop = True)
D_COL = 'join_date'
df[D_COL] = pd.to_datetime(df[D_COL], format = '%d/%m/%Y')
for f in ['month', 'year', 'week']:
    df[f] = getattr(df[D_COL].dt, f)
df['age'] = df['year'] - df['birth_year']


cat_cols = ['marital_status', 'branch_code', 'occupation_code', 'occupation_category_code', 'sex']
df[cat_cols] = df[cat_cols].apply(lambda x: pd.factorize(x)[0])
df['total_products_recommended'] = df[prod_cols].sum(axis = 1) # + 1
df['day_since_start'] = (df[D_COL] - df[D_COL].min()).dt.days

prod_cols_freq_sorted = df['target'].map(rev_mapper).value_counts().index.tolist()
prod_df = df[prod_cols].copy().astype('str')

In [13]:
for c in ['occupation_code', 'branch_code', 'birth_year', 'total_products_recommended', 'join_date']:
    df[f'n_unq_customers_in_{c}'] = df.groupby(c)['ID'].transform('nunique')

In [14]:
for p in tqdm_notebook(prod_cols_freq_sorted):
    df[f'branch_code_{p}_sum'] = df['branch_code'].map(df.groupby('branch_code')[p].sum())

In [15]:
for p in tqdm_notebook(prod_cols_freq_sorted):
    fltrd_data = df[df[p] == 1]
    df[f'occupation_code_{p}_nunq_customers'] = df['occupation_code'].map(fltrd_data.groupby('occupation_code')["ID"].nunique())
    df[f'branch_code_{p}_nunq_customers'] = df['branch_code'].map(fltrd_data.groupby('branch_code')["ID"].nunique())

In [16]:
df['branch_occupation_nunique'] = df['branch_code'].map(df.groupby('branch_code')['occupation_code'].nunique())
df['occupation_branch_nunique'] = df['occupation_code'].map(df.groupby('occupation_code')['branch_code'].nunique())

In [17]:
from itertools import combinations

cc = list(combinations(prod_cols,2))
tmp = pd.concat([df[c[1]].sub(df[c[0]]) for c in cc], axis=1, keys=cc)
tmp.columns = tmp.columns.map(''.join)
print (tmp.shape)
df = pd.concat([df, tmp], axis=1)

(76353, 210)


In [18]:
df.loc[~df['marital_status'].isin([0,1,2]), 'marital_status'] = np.nan


In [19]:
df['prod_comb'] = ''
for p in tqdm_notebook(prod_cols_freq_sorted):
    df['prod_comb'] = df['prod_comb'] + df[p].astype('str')
df['prod_comb'] = pd.factorize(df['prod_comb'])[0]

In [20]:
df['prod_comb_freq'] = df['prod_comb'].map(df['prod_comb'].value_counts(normalize=True))

In [21]:
vc = df['prod_comb'].value_counts()
less_freq_prod_comb = vc[vc < 5].index.tolist()
df.loc[df['prod_comb'].isin(less_freq_prod_comb), 'prod_comb'] = np.nan

In [22]:
df['join_date'] = pd.factorize(df['join_date'])[0]

In [23]:
df['birth_year_num'] = df['birth_year']
df['month_num'] = df['month']
df['week_num'] = df['week']
df['year_num'] = df['month']

In [24]:
features = [c for c in df.columns if c not in [ID_COL, TARGET_COL]]
cat_cols = ['join_date','sex','marital_status','birth_year','branch_code','occupation_code','occupation_category_code', 'prod_comb', 'month', 'year', 'week']
num_cols = [c for  c in features if c not in cat_cols]
df[cat_cols] = df[cat_cols].apply(lambda x: x.fillna(-999))

In [25]:
def clear_inf(x):
  x[(x == np.inf) | (x == -np.inf)] = np.nan
  return x
df[num_cols] = df[num_cols] = df[num_cols].apply(lambda x: clear_inf(x))
df[num_cols] = df[num_cols].apply(lambda x: x.fillna(x.mean() + 1e-6))
df[[c + '_log' for c in num_cols]] = df[num_cols].apply(lambda x: np.log1p(x))
num_cols = num_cols + [c + '_log' for c in num_cols]
df[num_cols] = df[num_cols] = df[num_cols].apply(lambda x: clear_inf(x))
df[num_cols] = df[num_cols].apply(lambda x: x.fillna(x.mean() + 1e-6))

scaler = MinMaxScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])
df[cat_cols] = df[cat_cols].astype('int')
df = pd.concat([df, pd.get_dummies(df[cat_cols], columns = cat_cols)], axis=1)

In [26]:
features = [c for c in df.columns if c not in [ID_COL, TARGET_COL]]
num_cols = [c for  c in features if c not in cat_cols]
len(num_cols)

1412

In [27]:
train, test = df[:train.shape[0]].reset_index(drop = True), df[train.shape[0]:].reset_index(drop = True)
train.shape, test.shape

((66353, 1425), (10000, 1425))

In [28]:
tmp = train[train['target'] == 20]
train = pd.concat([train, tmp], axis=0)

In [29]:
N_CLASSES = train['target'].nunique()
N_CLASSES

21

In [30]:
target = pd.get_dummies(train[TARGET_COL])
target.shape

(66357, 21)

In [31]:
def build_embedding_network(categorical_vars, num_cols, y_nunq, nunique_dict, max_embed_size=50):
    embeddings = []
    inputs = []

    for categorical_var in categorical_vars:    
        single_input = Input(shape=(1,))
        no_of_unique_cat  = nunique_dict[categorical_var]
        embedding_size = min(np.ceil((no_of_unique_cat)/2), max_embed_size)
        embedding_size = max(2, embedding_size)
        embedding_size = int(embedding_size)
        embedding = Embedding(no_of_unique_cat, embedding_size, input_length=1)(single_input)
        embedding = Reshape(target_shape=(embedding_size,))(embedding)
        inputs.append(single_input)
        embeddings.append(embedding)

    input_numeric = Input(shape=(len(num_cols),))
    inputs.append(input_numeric)
    embeddings.append(input_numeric)


    x = concatenate(embeddings)
    for i in range(3):
        x = Dense(512)(x)
        x =  PReLU()(x)
        x = Dropout(0.25)(x)
    output = Dense(21, activation='softmax')(x)
    model = Model(inputs, output)
    opt = Nadam(lr=1e-4)
    model.compile(loss='categorical_crossentropy', optimizer=opt)
    
    return model

In [32]:
es = tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                      patience = 12, 
                                      verbose=1, 
                                      restore_best_weights=True)

In [33]:
oofs = np.zeros((len(train), 21))
preds = np.zeros((len(test), 21))
fi_df = pd.DataFrame()

N_SPLITS = 5
folds = GroupKFold(n_splits = N_SPLITS)

for fold_, (trn_idx, val_idx) in enumerate(folds.split(train, target.values, train[ID_COL])):

  print(f'\n\n Fold {fold_} \n\n')
  X_trn, y_trn = train[features].iloc[trn_idx], target.iloc[trn_idx]
  X_val, y_val = train[features].iloc[val_idx], target.iloc[val_idx]
  X_tst = test[features]

  ### Adding categorical mappings
  nunique_dict = {c: X_trn[c].nunique() for c in cat_cols}
  cat_maps = {c: pd.Series(index = np.unique(X_trn[c]), data = np.arange(X_trn[c].nunique())) for c in cat_cols}
  proc_trn = [X_trn[c].map(cat_maps[c]).values for c in cat_cols]
  proc_val = [X_val[c].map(cat_maps[c]).values for c in cat_cols]
  proc_tst = [X_tst[c].map(cat_maps[c]).values for c in cat_cols]


  ### Adding numerical data
  proc_trn.append(X_trn[num_cols].values)
  proc_val.append(X_val[num_cols].values)
  proc_tst.append(X_tst[num_cols].values)
    
  clf = build_embedding_network(cat_cols, num_cols, N_CLASSES, nunique_dict, 100)

  gc.collect()

  print('\n\n ---- Starting NN Training ---- \n\n')
  history = clf.fit(proc_trn, y_trn, batch_size=256, epochs=100, validation_data = (proc_val, y_val), callbacks=[es])

  oofs[val_idx] = clf.predict(proc_val)

  val_score = log_loss(y_val, oofs[val_idx])
  print(f'\n\n Log loss for val is : {val_score}')

  preds += clf.predict(proc_tst) / N_SPLITS



 Fold 0 




0



 ---- Starting NN Training ---- 


Epoch 1/100
208/208 [==============================] - 2s 12ms/step - loss: 0.9936 - val_loss: 0.5661
Epoch 2/100
208/208 [==============================] - 2s 10ms/step - loss: 0.5660 - val_loss: 0.5056
Epoch 3/100
208/208 [==============================] - 2s 10ms/step - loss: 0.5019 - val_loss: 0.4528
Epoch 4/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4684 - val_loss: 0.4378
Epoch 5/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4502 - val_loss: 0.4370
Epoch 6/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4349 - val_loss: 0.4244
Epoch 7/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4214 - val_loss: 0.4108
Epoch 8/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4132 - val_loss: 0.4099
Epoch 9/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4023 - val_loss: 0.4026
Epoch 10/100
208/208 [=======================

9380



 ---- Starting NN Training ---- 


Epoch 1/100
208/208 [==============================] - 2s 11ms/step - loss: 0.9688 - val_loss: 0.5644
Epoch 2/100
208/208 [==============================] - 2s 10ms/step - loss: 0.5628 - val_loss: 0.4966
Epoch 3/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4998 - val_loss: 0.4628
Epoch 4/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4690 - val_loss: 0.4525
Epoch 5/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4457 - val_loss: 0.4445
Epoch 6/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4311 - val_loss: 0.4416
Epoch 7/100
208/208 [==============================] - 3s 13ms/step - loss: 0.4176 - val_loss: 0.4405
Epoch 8/100
208/208 [==============================] - 2s 12ms/step - loss: 0.4071 - val_loss: 0.4286
Epoch 9/100
208/208 [==============================] - 2s 11ms/step - loss: 0.3977 - val_loss: 0.4135
Epoch 10/100
208/208 [=======================

11149



 ---- Starting NN Training ---- 


Epoch 1/100
208/208 [==============================] - 3s 13ms/step - loss: 0.9815 - val_loss: 0.5631
Epoch 2/100
208/208 [==============================] - 2s 11ms/step - loss: 0.5667 - val_loss: 0.4898
Epoch 3/100
208/208 [==============================] - 2s 11ms/step - loss: 0.5051 - val_loss: 0.4584
Epoch 4/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4716 - val_loss: 0.4400
Epoch 5/100
208/208 [==============================] - 2s 12ms/step - loss: 0.4502 - val_loss: 0.4196
Epoch 6/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4352 - val_loss: 0.4143
Epoch 7/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4218 - val_loss: 0.4048
Epoch 8/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4137 - val_loss: 0.4022
Epoch 9/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4046 - val_loss: 0.3985
Epoch 10/100
208/208 [=======================

12351



 ---- Starting NN Training ---- 


Epoch 1/100
208/208 [==============================] - 2s 11ms/step - loss: 0.9697 - val_loss: 0.5749
Epoch 2/100
208/208 [==============================] - 2s 11ms/step - loss: 0.5675 - val_loss: 0.4780
Epoch 3/100
208/208 [==============================] - 2s 10ms/step - loss: 0.5009 - val_loss: 0.4458
Epoch 4/100
208/208 [==============================] - 3s 12ms/step - loss: 0.4721 - val_loss: 0.4331
Epoch 5/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4479 - val_loss: 0.4225
Epoch 6/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4321 - val_loss: 0.4187
Epoch 7/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4242 - val_loss: 0.4098
Epoch 8/100
208/208 [==============================] - 3s 16ms/step - loss: 0.4088 - val_loss: 0.4166
Epoch 9/100
208/208 [==============================] - 2s 12ms/step - loss: 0.4010 - val_loss: 0.4046
Epoch 10/100
208/208 [=======================

11136



 ---- Starting NN Training ---- 


Epoch 1/100
208/208 [==============================] - 2s 11ms/step - loss: 0.9844 - val_loss: 0.5929
Epoch 2/100
208/208 [==============================] - 4s 17ms/step - loss: 0.5632 - val_loss: 0.5009
Epoch 3/100
208/208 [==============================] - 2s 10ms/step - loss: 0.5001 - val_loss: 0.4709
Epoch 4/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4668 - val_loss: 0.4574
Epoch 5/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4452 - val_loss: 0.4462
Epoch 6/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4328 - val_loss: 0.4402
Epoch 7/100
208/208 [==============================] - 2s 11ms/step - loss: 0.4189 - val_loss: 0.4355
Epoch 8/100
208/208 [==============================] - 2s 10ms/step - loss: 0.4088 - val_loss: 0.4334
Epoch 9/100
208/208 [==============================] - 2s 10ms/step - loss: 0.3973 - val_loss: 0.4374
Epoch 10/100
208/208 [=======================

In [34]:
print(f'\nOOF val score: {log_loss(train[TARGET_COL], oofs)}')


OOF val score: 0.39823720898584003


In [35]:
oofs_df = pd.DataFrame(oofs).rename(rev_mapper, axis=1)
oofs_df['ID'] = train['ID'].values
oofs_df_melted = pd.melt(oofs_df, id_vars = 'ID', value_vars = prod_cols, var_name='prod', value_name = 'target')

In [36]:
train_cpy = train[['ID', 'target']]
train_cpy['target'] = train_cpy['target'].map(rev_mapper)
train_cpy = pd.get_dummies(train_cpy, columns = ['target'], prefix='', prefix_sep='')
train_cpy = train_cpy[oofs_df.columns.tolist()]
train_cpy_melted = pd.melt(train_cpy, id_vars = 'ID', value_vars = prod_cols, var_name='prod', value_name = 'target')
oofs_df_melted.loc[train_cpy_melted['target'] == 1, 'target'] = 1
log_loss(train_cpy_melted['target'], oofs_df_melted['target'])

0.011947990138152783

In [37]:
test_preds_df = pd.DataFrame(preds).rename(rev_mapper, axis=1)
test_preds_df['ID'] = test['ID'].values
test_preds_melted = pd.melt(test_preds_df, id_vars = 'ID', value_vars = prod_cols, var_name='prod', value_name = 'target')

In [38]:
test_preds_melted['ID X PCODE'] = test_preds_melted['ID'].astype('str') + ' X ' + test_preds_melted['prod']
mapper = pd.Series(index = test_preds_melted['ID X PCODE'], data = test_preds_melted['target'].values)

In [39]:
fltr = ss['Label'] != 1
ss.loc[fltr, 'Label'] = ss.loc[fltr, 'ID X PCODE'].map(mapper)
ss['Label'].describe().round(4)
ss.to_csv(f'm14_v4.csv', index=False)

count    210000.0000
mean          0.1088
std           0.2999
min           0.0000
25%           0.0000
50%           0.0000
75%           0.0018
max           1.0000
Name: Label, dtype: float64

In [40]:
ss.head(10)

,ID X PCODE,Label
0,F86J5PC X P5DA,0.000013
1,F86J5PC X RIBP,0.000046
2,F86J5PC X 8NN1,0.000003
3,F86J5PC X 7POT,0.000014
4,F86J5PC X 66FJ,0.000013
5,F86J5PC X GYSR,0.000001
6,F86J5PC X SOP4,0.000028
7,F86J5PC X RVSZ,1.000000
8,F86J5PC X PYUQ,0.000331
9,F86J5PC X LJR9,0.000023
